# Consumer Product Safety Project Capstone

This project analyzes consumer product-related injuries among children aged 0–17 in the United States in 2025. The analysis uses NEISS injury data, CPSC product recall data, and U.S. Census population estimates.

## 1. Import Libraries

I imported the Python libraries Pandas and numpy

In [ ]:
import pandas as pd
import numpy as np

## 2. Load Datasets

I loaded three datasets for this project: NEISS 2025 injury data, CPSC 2025 product recall data, and U.S. Census 2025 population estimates.

The CSV files are stored in Google Drive and loaded into Google Colab using pandas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
neiss = pd.read_csv('/content/drive/MyDrive/DATA205_Capstone/neiss2025.csv', low_memory=False)

recalls = pd.read_csv('/content/drive/MyDrive/DATA205_Capstone/product_recalls_download-1790116204.csv', low_memory=False)

population = pd.read_csv('/content/drive/MyDrive/DATA205_Capstone/nc-est2025-syasexn-census-population2025.csv', low_memory=False)

## 3. Datasets Inspection

Before cleaning the datasets, I examined their dimensions and structure using `.shape` and `.info()`.

- Datasets Dimensions


In [ ]:
print("NEISS:", neiss.shape)
print("Recalls:", recalls.shape)
print("Population:", population.shape)

NEISS: (410201, 25)
Recalls: (422, 22)
Population: (321, 8)


- Datasets structure

In [ ]:
print("NEISS DATASET")
neiss.info()

print("\nRECALLS DATASET")
recalls.info()

print("\nPOPULATION DATASET")
population.info()

NEISS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410201 entries, 0 to 410200
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CPSC_Case_Number   410201 non-null  int64  
 1   Treatment_Date     410201 non-null  object 
 2   Age                410201 non-null  int64  
 3   Sex                410201 non-null  int64  
 4   Race               410201 non-null  int64  
 5   Other_Race         15052 non-null   object 
 6   Hispanic           410201 non-null  int64  
 7   Body_Part          410201 non-null  int64  
 8   Diagnosis          410201 non-null  int64  
 9   Other_Diagnosis    101204 non-null  object 
 10  Body_Part_2        96775 non-null   float64
 11  Diagnosis_2        96775 non-null   float64
 12  Other_Diagnosis_2  27008 non-null   object 
 13  Disposition        410201 non-null  int64  
 14  Location           410201 non-null  int64  
 15  Fire_Involvement   410201 non-null  i

## 4. Data Cleaning

In this section, I clean and prepare the datasets for analysis.


### 4.1 NEISS 2025 Dataset

#### first rows

I am showing the first five rows of the NEISS dataset and examine its columns and values.




In [ ]:
neiss.head()

,CPSC_Case_Number,Treatment_Date,Age,Sex,Race,Other_Race,Hispanic,Body_Part,Diagnosis,Other_Diagnosis,...,Fire_Involvement,Product_1,Product_2,Product_3,Alcohol,Drug,Narrative_1,Stratum,PSU,Weight
0,250102066,1/1/2025,47,2,2,NaN,2,36,59,NaN,...,0,4076,1807,0,0,0,47YOF C/O RIGHT LEG LACERATION. WAS GETTING I...,M,200,46.4246
1,250102073,1/1/2025,64,1,1,NaN,2,37,64,NaN,...,0,1842,1829,0,0,0,64YOM C/O ANKLE PAIN AND RIGHT WRIST PAIN AFTE...,M,200,46.4246
2,250102084,1/1/2025,56,1,4,NaN,2,75,62,NaN,...,0,466,4056,0,0,0,56YOM C/O HEAD INJURY. LARGE POT FELL OFF A S...,M,200,46.4246
3,250102089,1/1/2025,73,1,1,NaN,2,76,59,NaN,...,0,4057,0,0,0,0,73YOM C/O FALL AND LACERATION. TRIPPED OVER H...,M,200,46.4246
4,250102091,1/1/2025,87,1,1,NaN,2,30,71,PAIN,...,0,676,0,0,0,0,87YOM C/O FALL ON BATHMAT. LEFT SHOULDER PAIN...,M,200,46.4246


#### Standardize Column Names

I converted the NEISS column names to lowercase to ensure consistency and make them easier to use during data cleaning and analysis.

In [ ]:
neiss.columns = neiss.columns.str.lower()
neiss.columns

Index(['cpsc_case_number', 'treatment_date', 'age', 'sex', 'race',
       'other_race', 'hispanic', 'body_part', 'diagnosis', 'other_diagnosis',
       'body_part_2', 'diagnosis_2', 'other_diagnosis_2', 'disposition',
       'location', 'fire_involvement', 'product_1', 'product_2', 'product_3',
       'alcohol', 'drug', 'narrative_1', 'stratum', 'psu', 'weight'],
      dtype='object')

#### Filter Children

I filtered the NEISS dataset to include only children aged 0–17.

In [ ]:
neiss_children = neiss.query('(age >= 2 and age <= 17) or (age >= 201 and age <= 223)').copy()
neiss_children.shape

(162894, 25)

#### Verifying Ages values


In [ ]:
neiss_children['age'].unique()

array([ 10,  11,   3,  14,  13,   2,  16,   4,   5, 218, 215,   9,   7,
       220, 214, 201, 206,  12,  17, 212, 202, 222,   8, 221, 213, 208,
       210,  15,   6, 217, 223, 203, 211, 207, 216, 209, 219, 205, 204])

#### Check Missing Values

I checked for missing values in the NEISS dataset.

In [ ]:
neiss_children.isna().sum()

,0
cpsc_case_number,0
treatment_date,0
age,0
sex,0
race,0
other_race,153502
hispanic,0
body_part,0
diagnosis,0
other_diagnosis,135656


#### Create Age Groups

I divided the children into four age groups (0–4, 5–10, 11–13, and 14–17) to compare injury patterns across different ages.

In [ ]:
neiss_children['age_group'] = np.where(
    (neiss_children['age'].between(2, 4)) | (neiss_children['age'].between(201, 223)),
    '0-4',
    np.where(
        neiss_children['age'].between(5, 10),
        '5-10',
        np.where(
            neiss_children['age'].between(11, 13),
            '11-13',
            '14-17'
        )
    )
)

#### Age Group Counts

I counted the injury records in each age group to verify the distribution of the NEISS data.

In [ ]:
neiss_children['age_group'].value_counts()

,count
age_group,
0-4,57638
5-10,45850
14-17,32947
11-13,26459


#### Decode Diagnoses

I used a dictionary to convert diagnosis codes into readable injury descriptions.
- for diagnosis

In [ ]:
diagnosis_codes = {
    41: 'Ingestion',
    42: 'Aspiration',
    46: 'Burns, Electrical',
    47: 'Burns, Not Specified',
    48: 'Burns, Scald',
    49: 'Burns, Chemical',
    50: 'Amputation',
    51: 'Burns, Thermal',
    52: 'Concussions',
    53: 'Contusions, Abrasions',
    54: 'Crushing',
    55: 'Dislocation',
    56: 'Foreign Body',
    57: 'Fracture',
    58: 'Hematoma',
    59: 'Laceration',
    60: 'Dental Injury',
    61: 'Nerve Damage',
    62: 'Internal Organ Injury',
    63: 'Puncture',
    64: 'Strain, Sprain',
    65: 'Anoxia',
    66: 'Hemorrhage',
    67: 'Electric Shock',
    68: 'Poisoning',
    69: 'Submersion',
    71: 'Other/Not Stated',
    72: 'Avulsion',
    73: 'Burns, Radiation',
    74: 'Dermatitis, Conjunctivitis'
}

In [ ]:
neiss_children = neiss_children.assign(
    diagnosis_decoded=lambda d: d['diagnosis'].map(diagnosis_codes)
)

- For diagnosis_2

In [ ]:
neiss_children = neiss_children.assign(
    diagnosis_2_decoded=lambda d: d['diagnosis_2'].map(diagnosis_codes)
)

Verifying the first 10 raws of the dataset

In [ ]:
neiss_children.head(10)

,cpsc_case_number,treatment_date,age,sex,race,other_race,hispanic,body_part,diagnosis,other_diagnosis,...,product_3,alcohol,drug,narrative_1,stratum,psu,weight,age_group,diagnosis_decoded,diagnosis_2_decoded
6,250102098,1/1/2025,10,2,2,NaN,2,76,59,NaN,...,1807,0,0,10YOF C/O FACIAL INJURY. WAS ASLEEP IN A CHAI...,M,200,46.4246,5-10,Laceration,NaN
7,250102113,1/1/2025,11,2,1,NaN,2,34,57,NaN,...,0,0,0,11YOF C/O ARM INJURY. WAS AT A TRAMPOLINE PAR...,M,200,46.4246,11-13,Fracture,NaN
10,250102205,1/1/2025,3,1,1,NaN,2,85,68,NaN,...,0,0,0,3YOM ARRIVES AFTER HIS FATHER'S VAPE FELL TO T...,S,97,101.4187,0-4,Poisoning,NaN
24,250102678,1/1/2025,14,1,1,NaN,2,34,57,NaN,...,0,0,0,14YOM C/O WRIST PAIN S/P HE STATES THAT HE FEL...,L,3,51.8171,14-17,Fracture,NaN
27,250102694,1/1/2025,13,1,1,NaN,2,37,71,INJURY,...,0,0,0,13YOM C/P ANKLE PAIN S/P PT ROLLED RIGHT ANKLE...,L,3,51.8171,11-13,Other/Not Stated,NaN
31,250102733,1/1/2025,2,1,3,BIRACIAL,2,76,59,NaN,...,0,0,0,2YOM C/O FACIAL LACERATION S/P MOTHER REPORTS ...,L,3,51.8171,0-4,Laceration,Internal Organ Injury
38,250102823,1/1/2025,16,2,1,NaN,2,83,71,PAIN,...,0,0,0,16YOF PRESENTS WITH RIGHT FOOT AND RIGHT ANKLE...,M,46,46.4246,14-17,Other/Not Stated,Other/Not Stated
41,250103131,1/1/2025,3,2,0,NaN,0,76,59,NaN,...,0,0,0,3YOF WITH AN EYELID LACERATION. AT APPROXIMATE...,C,18,7.1200,0-4,Laceration,NaN
42,250103133,1/1/2025,2,1,0,NaN,0,75,62,NaN,...,0,0,0,2YOM WITH A HEAD INJURY. MOM REPORTS PATIENT W...,C,18,7.1200,0-4,Internal Organ Injury,NaN
43,250103134,1/1/2025,4,2,0,NaN,0,76,59,NaN,...,0,0,0,4YOF WITH A CHIN LACERATION SUSTAINED AT HOME ...,C,18,7.1200,0-4,Laceration,NaN


Verifying if my variables have been created

In [ ]:
neiss_children.shape

(162894, 28)

#### Check Body Part Codes

I identified the unique body part codes in the NEISS dataset.


In [ ]:
neiss_children['body_part'].unique()

array([76, 34, 85, 37, 83, 75, 35, 77, 32, 89, 33, 88, 94,  0, 38, 92, 31,
       81, 36, 82, 93, 30, 87, 79, 80, 84])

#### Decode Body Parts

I converted the body part codes into readable descriptions to identify the body parts affected by injuries.
- For body_part

In [ ]:
body_part_codes = {
    0: 'Internal',
    30: 'Shoulder',
    31: 'Upper Trunk',
    32: 'Elbow',
    33: 'Lower Arm',
    34: 'Wrist',
    35: 'Knee',
    36: 'Lower Leg',
    37: 'Ankle',
    38: 'Pubic Region',
    75: 'Head',
    76: 'Face',
    77: 'Eyeball',
    79: 'Lower Trunk',
    80: 'Upper Arm',
    81: 'Upper Leg',
    82: 'Hand',
    83: 'Foot',
    84: '25-50% of Body',
    85: 'All Parts Body',
    87: 'Not Stated/Unknown',
    88: 'Mouth',
    89: 'Neck',
    92: 'Finger',
    93: 'Toe',
    94: 'Ear'
}

In [ ]:
neiss_children = neiss_children.assign(
    body_part_decoded=lambda d: d['body_part'].map(body_part_codes)
)

In [ ]:
neiss_children[['body_part', 'body_part_decoded']].head()

,body_part,body_part_decoded
6,76,Face
7,34,Wrist
10,85,All Parts Body
24,34,Wrist
27,37,Ankle


- For body_part2

In [ ]:
neiss_children = neiss_children.assign(
    body_part_2_decoded=lambda d: d['body_part_2'].map(body_part_codes)
)

In [ ]:
neiss_children[['body_part_2', 'body_part_2_decoded']].head(10)

,body_part_2,body_part_2_decoded
6,NaN,NaN
7,NaN,NaN
10,NaN,NaN
24,NaN,NaN
27,NaN,NaN
31,75.0,Head
38,37.0,Ankle
41,NaN,NaN
42,NaN,NaN
43,NaN,NaN


#### Decode Disposition

I converted disposition codes into readable descriptions to understand patient outcomes after emergency department visits.

In [ ]:
disposition_codes = {
    0: 'No Injury',
    1: 'Treated and Released',
    2: 'Treated and Transferred',
    4: 'Hospitalized',
    5: 'Held for Observation',
    6: 'Left Without Being Seen',
    8: 'Fatality',
    9: 'Unknown'
}

In [ ]:
neiss_children = neiss_children.assign(
    disposition_decoded=lambda d: d['disposition'].map(disposition_codes)
)

In [ ]:
neiss_children[['disposition', 'disposition_decoded']].head(10)

,disposition,disposition_decoded
6,1,Treated and Released
7,1,Treated and Released
10,1,Treated and Released
24,1,Treated and Released
27,1,Treated and Released
31,1,Treated and Released
38,1,Treated and Released
41,1,Treated and Released
42,1,Treated and Released
43,1,Treated and Released


#### Classify Injury Severity

I classified injury outcomes as Severe, Not Severe, or Unknown using NEISS disposition codes.

In [ ]:
neiss_children = neiss_children.assign(
    severity=lambda d: np.where(
        d['disposition'].isin([2, 4, 8]),
        'Severe',
        np.where(
            d['disposition'] == 9,
            'Unknown',
            'Not Severe'
        )
    )
)

#### Product Code Cleaning and Decoding


In [ ]:
neiss_codes = pd.read_excel(
    '/content/drive/MyDrive/DATA205_Capstone/neiss2025.xlsx',
    sheet_name='NEISS_FMT'
)

neiss_codes.head()

,Format name,Starting value for format,Ending value for format,Format value label
0,AGELTTWO,0,0,UNK
1,AGELTTWO,2,120,2 YEARS AND OLDER
2,AGELTTWO,201,201,1 MONTH
3,AGELTTWO,202,202,2 MONTHS
4,AGELTTWO,203,203,3 MONTHS


In [ ]:
product_codes = neiss_codes[
    neiss_codes['Format name'] == 'PROD'
]

product_codes.head(10)

,Format name,Starting value for format,Ending value for format,Format value label
116,PROD,101,101,101 - WASHING MACHINES WITHOUT WRINGERS OR OTH...
117,PROD,102,102,102 - WRINGER WASHING MACHINES
118,PROD,103,103,103 - WASHING MACHINES WITH UNHEATED SPIN DRYERS
119,PROD,106,106,106 - ELECTRIC CLOTHES DRYERS WITHOUT WASHERS
120,PROD,107,107,107 - GAS CLOTHES DRYERS WITHOUT WASHERS
121,PROD,108,108,108 - MANGLE IRONS
122,PROD,110,110,110 - ELECTRIC HEATING PADS
123,PROD,112,112,112 - SEWING MACHINES OR ACCESSORIES
124,PROD,113,113,113 - FLOOR BUFFERS OR WAXERS
125,PROD,114,114,114 - RUG SHAMPOOERS


In [ ]:
product_dict = dict(zip(
    product_codes['Starting value for format'],
    product_codes['Format value label']
))

list(product_dict.items())[:10]

[('             101',
  '101 - WASHING MACHINES WITHOUT WRINGERS OR OTHER DRYERS'),
 ('             102', '102 - WRINGER WASHING MACHINES'),
 ('             103', '103 - WASHING MACHINES WITH UNHEATED SPIN DRYERS'),
 ('             106', '106 - ELECTRIC CLOTHES DRYERS WITHOUT WASHERS'),
 ('             107', '107 - GAS CLOTHES DRYERS WITHOUT WASHERS'),
 ('             108', '108 - MANGLE IRONS'),
 ('             110', '110 - ELECTRIC HEATING PADS'),
 ('             112', '112 - SEWING MACHINES OR ACCESSORIES'),
 ('             113', '113 - FLOOR BUFFERS OR WAXERS'),
 ('             114', '114 - RUG SHAMPOOERS')]

In [ ]:
# Select and clean product codes
product_codes = neiss_codes[neiss_codes['Format name'] == 'PROD'].copy()

product_codes['Starting value for format'] = pd.to_numeric(
    product_codes['Starting value for format'].str.strip(),
    errors='coerce'
)

# Create product dictionary
product_dict = dict(zip(
    product_codes['Starting value for format'],
    product_codes['Format value label']
))

# Decode the three product columns
neiss_children = neiss_children.assign(
    product_1_decoded=lambda d: d['product_1'].map(product_dict),
    product_2_decoded=lambda d: d['product_2'].map(product_dict),
    product_3_decoded=lambda d: d['product_3'].map(product_dict)
)

# Check results
neiss_children[[
    'product_1', 'product_1_decoded',
    'product_2', 'product_2_decoded',
    'product_3', 'product_3_decoded'
]].head(10)

,product_1,product_1_decoded,product_2,product_2_decoded,product_3,product_3_decoded
6,4074,"4074 - CHAIRS, OTHER OR NOT SPECIFIED",1715,1715 - PET SUPPLIES,1807,1807 - FLOORS OR FLOORING MATERIALS
7,1233,1233 - TRAMPOLINES,0,NaN,0,NaN
10,1807,1807 - FLOORS OR FLOORING MATERIALS,0,NaN,0,NaN
24,5031,"5031 - SNOWBOARDING (ACTIVITY, APPAREL OR EQUI...",0,NaN,0,NaN
27,1233,1233 - TRAMPOLINES,0,NaN,0,NaN
31,4056,"4056 - CABINETS, RACKS, ROOM DIVIDERS AND SHEL...",0,NaN,0,NaN
38,4074,"4074 - CHAIRS, OTHER OR NOT SPECIFIED",0,NaN,0,NaN
41,4074,"4074 - CHAIRS, OTHER OR NOT SPECIFIED",1807,1807 - FLOORS OR FLOORING MATERIALS,0,NaN
42,4074,"4074 - CHAIRS, OTHER OR NOT SPECIFIED",0,NaN,0,NaN
43,1842,1842 - STAIRS OR STEPS,0,NaN,0,NaN


#### Save Cleaned NEISS Dataset


In [ ]:
neiss_children.to_csv(
    '/content/drive/MyDrive/DATA205_Capstone/neiss_children_clean.csv',
    index=False
)

### 4.2 CPSC Product Recalls 2025 Dataset



In [ ]:
recalls.head()

,Recall Number,Product Safety Warning Number,Date,Product Safety Warning Date,Recall Heading,Name of product,Description,Hazard Description,Consumer Action,Original Product Safety Warning Announcement,...,Incidents,Remedy,Sold At Label,Sold At,Importers,Manufacturers,Distributors,Manufactured In,Custom Label,Custom Field
0,26-164,NaN,"December 18, 2025",NaN,Meaicezli Play Purse Sets Recalled Due to Risk...,Meaicezli Play Purse Sets,This recall involves Meaicezli Play Purse Sets...,The children’s toy violates the mandatory stan...,NaN,NaN,...,None reported,Consumers should stop using the toy purse set ...,Sold Online At,Amazon.com from August 2025 through October 20...,NaN,NaN,NaN,China,Retailer:,Qianshanshiangeerdianzishangwuyouxiangongsi do...
1,26-160,NaN,"December 18, 2025",NaN,JOKOSIS Adult Portable Bed Rails Recalled Due ...,JOKOSIS Portable Bed Rails,"This recall involves JOKOSIS Bed Rail, model H...",The recalled bed rails violate the mandatory s...,NaN,NaN,...,None reported,Consumers should immediately stop using the JO...,Sold Online At,Amazon.com from August 2025 through November 2...,NaN,NaN,NaN,China,Retailer:,"Guangshui Xie Lan Zheng Trading Co., Ltd.,&nbs..."
2,26-163,NaN,"December 18, 2025",NaN,Blue Wave Recalls Above-Ground Pools 48 Inches...,Blue Wave brand 48-inch and taller above-groun...,This recall involves all Blue Wave brand above...,The compression strap that surrounds the outsi...,NaN,NaN,...,None reported,Consumers should contact Blue Wave to request ...,Sold At,"The Home Depot, Lowe’s, and Dunham’s Sports re...","Blue Wave Products Inc., of Batavia, Illinois",NaN,NaN,China,NaN,NaN
3,26-162,NaN,"December 18, 2025",NaN,SereneLife Home Recalls Above-Ground Pools 48 ...,SereneLife brand 48-inch and taller above-grou...,This recall involves all SereneLife brand abov...,The compression strap that surrounds the outsi...,NaN,NaN,...,None reported,Consumers should stop using the recalled above...,Sold Online At,"Amazon.com, Walmart.com, Wayfair.com, Target.c...",Sound Around Inc. dba SereneLife Home,NaN,NaN,China,NaN,NaN
4,26-159,NaN,"December 18, 2025",NaN,KKL Fabric 9-Drawer Dressers Recalled Due to R...,KKL Fabric 9-Drawer Dressers,This recall involves KKL Fabric 9-Drawer Dress...,The recalled dressers are unstable if they are...,NaN,NaN,...,None reported,Consumers should stop using the recalled dress...,Sold Online At,Amazon.com from September 2023 through Novembe...,NaN,Changzhou Municipal Sea Metal Products Co. Ltd...,NaN,China,Retailer:,"HK Brilliant dba KKL, of China"


copying my proceed data st to the drive

#### Create a Working Copy

I created a copy of the original recalls dataset to preserve the raw data during cleaning.

In [ ]:
recalls_clean = recalls.copy()

#### Standardize Column Names

I remove all upper cases to the column names to make them easier to use in Python and replacing space between the name variable by the _

In [ ]:
recalls_clean.columns = recalls_clean.columns.str.lower().str.replace(' ', '_')
recalls_clean.head()


,recall_number,product_safety_warning_number,date,product_safety_warning_date,recall_heading,name_of_product,description,hazard_description,consumer_action,original_product_safety_warning_announcement,...,incidents,remedy,sold_at_label,sold_at,importers,manufacturers,distributors,manufactured_in,custom_label,custom_field
0,26-164,NaN,"December 18, 2025",NaN,Meaicezli Play Purse Sets Recalled Due to Risk...,Meaicezli Play Purse Sets,This recall involves Meaicezli Play Purse Sets...,The children’s toy violates the mandatory stan...,NaN,NaN,...,None reported,Consumers should stop using the toy purse set ...,Sold Online At,Amazon.com from August 2025 through October 20...,NaN,NaN,NaN,China,Retailer:,Qianshanshiangeerdianzishangwuyouxiangongsi do...
1,26-160,NaN,"December 18, 2025",NaN,JOKOSIS Adult Portable Bed Rails Recalled Due ...,JOKOSIS Portable Bed Rails,"This recall involves JOKOSIS Bed Rail, model H...",The recalled bed rails violate the mandatory s...,NaN,NaN,...,None reported,Consumers should immediately stop using the JO...,Sold Online At,Amazon.com from August 2025 through November 2...,NaN,NaN,NaN,China,Retailer:,"Guangshui Xie Lan Zheng Trading Co., Ltd.,&nbs..."
2,26-163,NaN,"December 18, 2025",NaN,Blue Wave Recalls Above-Ground Pools 48 Inches...,Blue Wave brand 48-inch and taller above-groun...,This recall involves all Blue Wave brand above...,The compression strap that surrounds the outsi...,NaN,NaN,...,None reported,Consumers should contact Blue Wave to request ...,Sold At,"The Home Depot, Lowe’s, and Dunham’s Sports re...","Blue Wave Products Inc., of Batavia, Illinois",NaN,NaN,China,NaN,NaN
3,26-162,NaN,"December 18, 2025",NaN,SereneLife Home Recalls Above-Ground Pools 48 ...,SereneLife brand 48-inch and taller above-grou...,This recall involves all SereneLife brand abov...,The compression strap that surrounds the outsi...,NaN,NaN,...,None reported,Consumers should stop using the recalled above...,Sold Online At,"Amazon.com, Walmart.com, Wayfair.com, Target.c...",Sound Around Inc. dba SereneLife Home,NaN,NaN,China,NaN,NaN
4,26-159,NaN,"December 18, 2025",NaN,KKL Fabric 9-Drawer Dressers Recalled Due to R...,KKL Fabric 9-Drawer Dressers,This recall involves KKL Fabric 9-Drawer Dress...,The recalled dressers are unstable if they are...,NaN,NaN,...,None reported,Consumers should stop using the recalled dress...,Sold Online At,Amazon.com from September 2023 through Novembe...,NaN,Changzhou Municipal Sea Metal Products Co. Ltd...,NaN,China,Retailer:,"HK Brilliant dba KKL, of China"


#### Check Missing Values

I checked for missing values.

In [ ]:
pd.isna(recalls_clean).sum()

,0
recall_number,0
product_safety_warning_number,419
date,0
product_safety_warning_date,419
recall_heading,0
name_of_product,0
description,0
hazard_description,0
consumer_action,420
original_product_safety_warning_announcement,419


In [ ]:
recalls_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 422 entries, 0 to 421
Data columns (total 22 columns):
 #   Column                                        Non-Null Count  Dtype 
---  ------                                        --------------  ----- 
 0   recall_number                                 422 non-null    object
 1   product_safety_warning_number                 3 non-null      object
 2   date                                          422 non-null    object
 3   product_safety_warning_date                   3 non-null      object
 4   recall_heading                                422 non-null    object
 5   name_of_product                               422 non-null    object
 6   description                                   422 non-null    object
 7   hazard_description                            422 non-null    object
 8   consumer_action                               2 non-null      object
 9   original_product_safety_warning_announcement  3 non-null      object
 10  re

Checking format's date

In [ ]:
recalls_clean['date'].unique()

array(['December 18, 2025', 'December 11, 2025', 'December 05, 2025',
       'December 04, 2025', 'November 26, 2025', 'November 20, 2025',
       'November 13, 2025', 'November 06, 2025', 'October 30, 2025',
       'October 23, 2025', 'October 16, 2025', 'October 09, 2025',
       'October 02, 2025', 'September 25, 2025', 'September 18, 2025',
       'September 11, 2025', 'September 04, 2025', 'August 28, 2025',
       'August 21, 2025', 'August 14, 2025', 'August 07, 2025',
       'July 31, 2025', 'July 24, 2025', 'July 21, 2025', 'July 17, 2025',
       'July 10, 2025', 'July 03, 2025', 'June 26, 2025', 'June 18, 2025',
       'June 12, 2025', 'June 05, 2025', 'May 29, 2025', 'May 22, 2025',
       'May 15, 2025', 'May 08, 2025', 'May 01, 2025', 'April 24, 2025',
       'April 17, 2025', 'April 10, 2025', 'April 03, 2025',
       'March 27, 2025', 'March 20, 2025', 'March 13, 2025',
       'March 06, 2025', 'February 27, 2025', 'February 20, 2025',
       'February 13, 2025', 'Febru

#### Convert Date Column

I converted the recall dates into datetime format to support monthly recall analysis. because the actuel format is '%B %d, %Y' example December 15,2025 i need this format %Y''%m %d. 2025-12-15 which is this type: datetime64[ns]

In [ ]:
recalls_clean['date'] = pd.to_datetime(
    recalls_clean['date'],
    format='%B %d, %Y',
    errors='coerce'
)

recalls_clean['date'].head()

,date
0,2025-12-18
1,2025-12-18
2,2025-12-18
3,2025-12-18
4,2025-12-18


#### Inspect Country of Manufacture


In [ ]:
print(recalls_clean['manufactured_in'].unique())

['China' 'United States' 'India' nan 'Cambodia' 'Taiwan' 'Turkey'
 'Vietnam (MMA008) and China (BPB002 and PB0003)' 'Peru' 'Italy'
 'Indonesia' 'China, Portugal' 'Vietnam' 'Canada' 'Korea' 'Japan'
 'USA, Poland and Taiwan' 'Vietnam and China' 'Thailand' 'Bangladesh'
 'Honduras' 'Cambodia, China and Vietnam' 'Mexico' 'Switzerland'
 'Colombia' 'Czech Republic' 'United States and Japan'
 'China and Thailand' 'Austria' 'China and Vietnam' 'Vietnam and Taiwan'
 'China and Malaysia' 'Canada (ski kits)\r\nChina (adaptor kits)'
 'Germany' 'Philippines' 'Peru and Portugal' 'Austria and Bosnia' 'France'
 'China, Hong Kong and Vietnam' 'Belgium' 'United States and China'
 'Malaysia' 'Mexico and Korea' 'Vietnam, China, and Taiwan']


In [ ]:
recalls_clean['countries_clean'] = (
    recalls_clean['manufactured_in']
    .str.replace(' and ', ', ')
    .str.split(', ')
)

recalls_clean[['manufactured_in', 'countries_clean']].head(20)

,manufactured_in,countries_clean
0,China,[China]
1,China,[China]
2,China,[China]
3,China,[China]
4,China,[China]
5,China,[China]
6,United States,[United States]
7,China,[China]
8,China,[China]
9,India,[India]


In [ ]:
recalls_clean[['manufactured_in', 'countries_clean']].tail(20)

,manufactured_in,countries_clean
402,China,[China]
403,Taiwan,[Taiwan]
404,China,[China]
405,China,[China]
406,Italy,[Italy]
407,Cambodia,[Cambodia]
408,China,[China]
409,United States,[United States]
410,China,[China]
411,China,[China]


#### Inspect Hazard Descriptions


In [ ]:
recalls_clean['hazard_description'].head(10)

,hazard_description
0,The children’s toy violates the mandatory stan...
1,The recalled bed rails violate the mandatory s...
2,The compression strap that surrounds the outsi...
3,The compression strap that surrounds the outsi...
4,The recalled dressers are unstable if they are...
5,"The beard serum contains minoxidil, which must..."
6,"The ointment contains lidocaine, which must be..."
7,The power strips do not contain supplementary ...
8,The recalled dressers are unstable if they are...
9,The component that connects the recalled chand...


In [ ]:
recalls_clean['hazard_description'].unique()

array(['The children’s toy violates the mandatory standard for&nbsp;https://www.cpsc.gov/Business--Manufacturing/Business-Education/Toy-Saf…; because the toy cell phone contains button cell batteries and the toy tablet contains a lithium coin battery that can be easily accessed by children. When button cell or coin batteries are swallowed, the ingested batteries can cause serious injuries, internal chemical burns and death.',
       'The recalled bed rails violate the mandatory standard for&nbsp;https://www.cpsc.gov/Business--Manufacturing/Business-Education/Busines… portable bed rails because when the bed rails are attached to a bed, users can become entrapped within the bed rail or between the bed rail and the side of the mattress, posing a serious entrapment hazard and risk of death by asphyxiation. In addition, the bed rails do not bear the required hazard warning labels.',
       'The compression strap that surrounds the outside of the pool legs may create a foothold, allowing a c

#### Refine Hazard Keywords

In [ ]:
hazard_keywords = {}
hazard_keywords['Laceration'] = ['laceration']
hazard_keywords['Crash'] = ['crash', 'collision', 'brake failure']
hazard_keywords['Impact'] = ['impact hazard', 'impact injury', 'injury from impact']

hazard_keywords['Overheating'] = ['overheat', 'overheating']
hazard_keywords['Head Injury'] = ['head injury']
hazard_keywords['Skin Irritation'] = ['skin irritation', 'respiratory irritation']
hazard_keywords['Infection'] = ['infection', 'bacteria', 'mold exposure']

In [ ]:
for hazard, keywords in hazard_keywords.items():
    recalls_clean[hazard] = False

    for word in keywords:
        recalls_clean[hazard] = (
            recalls_clean[hazard] |
            recalls_clean['hazard_description'].str.contains(
                word, case=False, na=False, regex=False
            )
        )

In [ ]:
hazard_counts = recalls_clean[list(hazard_keywords.keys())].sum()

hazard_counts.sort_values(ascending=False)


,0
Overheating,45
Laceration,29
Crash,25
Impact,9
Infection,8
Head Injury,6
Skin Irritation,2


In [ ]:
hazard_counts = recalls_clean[list(hazard_keywords.keys())].sum().sort_values(ascending=False)

hazard_counts

,0
Overheating,45
Laceration,29
Crash,25
Impact,9
Infection,8
Head Injury,6
Skin Irritation,2


In [ ]:
hazard_columns = list(hazard_keywords.keys())

unclassified = recalls_clean[
    ~recalls_clean[hazard_columns].any(axis=1)
]

unclassified.shape[0]

307

In [ ]:
for hazard in unclassified['hazard_description']:
    print(hazard)
    print()

The children’s toy violates the mandatory standard for&nbsp;https://www.cpsc.gov/Business--Manufacturing/Business-Education/Toy-Saf…; because the toy cell phone contains button cell batteries and the toy tablet contains a lithium coin battery that can be easily accessed by children. When button cell or coin batteries are swallowed, the ingested batteries can cause serious injuries, internal chemical burns and death.

The recalled bed rails violate the mandatory standard for&nbsp;https://www.cpsc.gov/Business--Manufacturing/Business-Education/Busines… portable bed rails because when the bed rails are attached to a bed, users can become entrapped within the bed rail or between the bed rail and the side of the mattress, posing a serious entrapment hazard and risk of death by asphyxiation. In addition, the bed rails do not bear the required hazard warning labels.

The compression strap that surrounds the outside of the pool legs may create a foothold, allowing a child access to the pool, p

In [ ]:
hazard_keywords['Pressure / Rupture'] = ['excessive pressure', 'rupture']
hazard_keywords['Collapse'] = ['collapse']
hazard_keywords['Structural Failure'] = ['welding can break']
hazard_keywords['Circulation Restriction'] = ['restrict blood circulation']
hazard_keywords['Rollover'] = ['rolls over', 'rollover']
hazard_keywords['Equipment Failure'] = ['malfunction', 'loss of emergency communication']
hazard_keywords['Unexpected Activation'] = ['starts the motorcycle unexpectedly']

Classify Recall Hazards

In [ ]:
for hazard, keywords in hazard_keywords.items():
    recalls_clean[hazard] = False

    for word in keywords:
        recalls_clean[hazard] = (
            recalls_clean[hazard] |
            recalls_clean['hazard_description'].str.contains(
                word, case=False, na=False, regex=False
            )
        )

Count Recalls by Hazard Category

In [ ]:
hazard_counts = (
    recalls_clean[list(hazard_keywords.keys())]
    .sum()
    .sort_values(ascending=False)
)

hazard_counts

,0
Overheating,45
Laceration,29
Crash,25
Impact,9
Infection,8
Head Injury,6
Equipment Failure,6
Collapse,4
Skin Irritation,2
Pressure / Rupture,1


#### Save Cleaned CPSC Recalls Dataset

Save the cleaned CPSC recalls dataset

In [ ]:
recalls_clean.to_csv(
    '/content/drive/MyDrive/DATA205_Capstone/recalls_clean.csv',
    index=False
)

### Census Population Data Cleaning

#### Inspect Census Dataset

Examine the structure, columns, and first rows of the Census population dataset.

In [ ]:
print(population.shape)
population.head()

(321, 8)


,table with row headers in column A and column headers in rows 3 through 4. (leading dots indicate sub-parts),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,Annual Estimates of the Resident Population by...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sex and Age,"April 1, 2020\nEstimates Base",Population Estimate (as of July 1),NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2020,2021,2022,2023,2024,2025
3,Total Population,"331,516,113","331,578,104","332,100,166","333,996,304","336,755,052","340,003,797","341,784,857"
4,.0,"3,719,470","3,691,723","3,572,165","3,680,529","3,653,453","3,617,221","3,618,898"


#### Create a Copy of Census Dataset

Create a copy of the original Census dataset for cleaning.

In [ ]:
population_clean = population.copy()

Check missing values

In [ ]:
# Check missing values
population_clean.isna().sum()

,0
table with row headers in column A and column headers in rows 3 through 4. (leading dots indicate sub-parts),4
Unnamed: 1,11
Unnamed: 2,10
Unnamed: 3,11
Unnamed: 4,11
Unnamed: 5,11
Unnamed: 6,11
Unnamed: 7,11


These missing Values are for the title and separation betweens sections so we can drop it.

In [ ]:
# Remove missing values
population_clean = population_clean.dropna()

# Check missing values
population_clean.isna().sum()

,0
table with row headers in column A and column headers in rows 3 through 4. (leading dots indicate sub-parts),0
Unnamed: 1,0
Unnamed: 2,0
Unnamed: 3,0
Unnamed: 4,0
Unnamed: 5,0
Unnamed: 6,0
Unnamed: 7,0


giving appropriate names to the column now the are unnamed

In [ ]:
# Rename columns
population_clean.columns = [
    'Age',
    'Estimates_Base_2020',
    'Population_2020',
    'Population_2021',
    'Population_2022',
    'Population_2023',
    'Population_2024',
    'Population_2025'
]

# Check column names
population_clean.columns

Index(['Age', 'Estimates_Base_2020', 'Population_2020', 'Population_2021',
       'Population_2022', 'Population_2023', 'Population_2024',
       'Population_2025'],
      dtype='object')

In [ ]:
# Remove commas and convert columns to numeric
for col in population_clean.columns[1:]:
    population_clean[col] = pd.to_numeric(
        population_clean[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )



In [ ]:
# Clean Age column
population_clean['Age'] = population_clean['Age'].str.lstrip('.')


Saving the Cleaned Dataset to Google Drive

In [ ]:
# Save cleaned Census population dataset
population_clean.to_csv("population_2025_cleaned.csv", index=False)

selecting childrens for the 2025 year

In [ ]:
# Select children aged 0-17
population_children = population_clean.iloc[1:19].copy()

# Keep only Age and Population_2025
population_children = population_children[
    ['Age', 'Population_2025']
]

# Display the result
display(population_children)

,Age,Population_2025
4,0,3618898.0
5,1,3642657.0
6,2,3725891.0
7,3,3791993.0
8,4,3712428.0
9,5,3833003.0
10,6,3887513.0
11,7,3958385.0
12,8,4013109.0
13,9,4095711.0


Children dataset

In [ ]:
population_children.to_csv(
    '/content/drive/MyDrive/population_children_2025_processed.csv',
    index=False
)